In [10]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from src.spark_session import get_spark_session

spark = get_spark_session(
    app_name="Silver-Transformation",
    master="local[2]",
    driver_memory="3g"
)

print(f"✅ Spark {spark.version}")

✅ Spark 3.5.1


# 🥈 Silver Katmanı: Veri Temizleme ve Doğrulama

Bronze tablosundan oku, **temizle ve zenginleştir**, Silver'a yaz.

## Yapılan İşlemler

1. **Tip dönüşümü**: `timestamp`, `invoice_date` string'den TIMESTAMP'e
2. **Temizlik**:
   - CustomerID null olanları çıkar (ML için kullanılamaz)
   - Description null olanları çıkar
   - Duplikat kayıtları kaldır
3. **Türetilmiş alanlar**:
   - `total_price = quantity * unit_price`
   - `is_cancellation` (boolean flag)
4. **Ayrım**:
   - `silver/transactions` → normal alışverişler
   - `silver/cancellations` → iptaller (analiz için ayrı tutulur)

In [11]:
BRONZE_PATH = os.path.abspath("../delta_lake/bronze/transactions")

df_bronze = spark.read.format("delta").load(BRONZE_PATH)
print(f"📊 Bronze: {df_bronze.count():,} satır")

df_bronze.printSchema()

📊 Bronze: 10,100 satır
root
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- kullanici_ID: long (nullable = true)
 |-- olay_tipi: string (nullable = true)
 |-- ilgili_ID: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- country: string (nullable = true)
 |-- invoice_date: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)



In [12]:
from pyspark.sql.functions import col, to_timestamp, when, expr

df_typed = df_bronze \
    .withColumn("timestamp", to_timestamp(col("timestamp"))) \
    .withColumn("invoice_date", to_timestamp(col("invoice_date"))) \
    .withColumn("total_price", col("quantity") * col("unit_price")) \
    .withColumn("is_cancellation", col("olay_tipi") == "cancellation")

print("✅ Tipler dönüştürüldü, türetilmiş alanlar eklendi")
df_typed.select("timestamp", "invoice_date", "quantity", "unit_price", "total_price", "is_cancellation").show(5)

✅ Tipler dönüştürüldü, türetilmiş alanlar eklendi
+--------------------+-------------------+--------+----------+------------------+---------------+
|           timestamp|       invoice_date|quantity|unit_price|       total_price|is_cancellation|
+--------------------+-------------------+--------+----------+------------------+---------------+
|2026-05-09 18:12:...|2009-12-01 07:45:00|      12|      6.95|              83.4|          false|
|2026-05-09 18:12:...|2009-12-01 07:45:00|      12|      6.75|              81.0|          false|
|2026-05-09 18:12:...|2009-12-01 07:45:00|      12|      6.75|              81.0|          false|
|2026-05-09 18:12:...|2009-12-01 07:45:00|      48|       2.1|100.80000000000001|          false|
|2026-05-09 18:12:...|2009-12-01 07:45:00|      24|      1.25|              30.0|          false|
+--------------------+-------------------+--------+----------+------------------+---------------+
only showing top 5 rows



In [13]:
print(f"📊 Başlangıç: {df_typed.count():,} satır")

# 1. CustomerID null olanları çıkar
df_clean = df_typed.filter(col("kullanici_ID").isNotNull())
print(f"   ↳ CustomerID null silindikten sonra: {df_clean.count():,} satır")

# 2. Description null olanları çıkar
df_clean = df_clean.filter(col("description").isNotNull())
print(f"   ↳ Description null silindikten sonra: {df_clean.count():,} satır")

# 3. Duplikatları kaldır (aynı invoice + stock + customer + timestamp)
before = df_clean.count()
df_clean = df_clean.dropDuplicates(["ilgili_ID", "stock_code", "kullanici_ID", "invoice_date", "quantity"])
print(f"   ↳ Duplikatlar kaldırıldıktan sonra: {df_clean.count():,} satır (kaldırılan: {before - df_clean.count()})")

# 4. Anormal Quantity ve UnitPrice (purchase'larda 0 veya negatif olamaz)
# Cancellation'larda quantity zaten negatif olabilir, onları ayıracağız
print(f"\n📊 Final temizlenmiş Silver: {df_clean.count():,} satır")

📊 Başlangıç: 10,100 satır
   ↳ CustomerID null silindikten sonra: 7,295 satır
   ↳ Description null silindikten sonra: 7,295 satır
   ↳ Duplikatlar kaldırıldıktan sonra: 7,104 satır (kaldırılan: 191)

📊 Final temizlenmiş Silver: 7,104 satır


In [15]:
df_purchases = df_clean.filter(col("is_cancellation") == False) \
                       .filter(col("quantity") > 0) \
                       .filter(col("unit_price") > 0)

df_cancellations = df_clean.filter(col("is_cancellation") == True)

print(f"🛒 Normal alışverişler:  {df_purchases.count():,} satır")
print(f"❌ İptaller:              {df_cancellations.count():,} satır")

🛒 Normal alışverişler:  6,932 satır
❌ İptaller:              170 satır


In [16]:
SILVER_PURCHASES_PATH = os.path.abspath("../delta_lake/silver/transactions")
SILVER_CANCELLATIONS_PATH = os.path.abspath("../delta_lake/silver/cancellations")

# Purchase'ları yaz
df_purchases.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_PURCHASES_PATH)

print(f"✅ Silver purchases yazıldı: {SILVER_PURCHASES_PATH}")

# Cancellation'ları yaz
df_cancellations.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_CANCELLATIONS_PATH)

print(f"✅ Silver cancellations yazıldı: {SILVER_CANCELLATIONS_PATH}")

✅ Silver purchases yazıldı: /Users/tunakomur/Desktop/big-data-online-retail-pipeline/delta_lake/silver/transactions
✅ Silver cancellations yazıldı: /Users/tunakomur/Desktop/big-data-online-retail-pipeline/delta_lake/silver/cancellations


In [17]:
df_silver_purchases = spark.read.format("delta").load(SILVER_PURCHASES_PATH)
df_silver_cancellations = spark.read.format("delta").load(SILVER_CANCELLATIONS_PATH)

print(f"🛒 Silver purchases:    {df_silver_purchases.count():,} satır")
print(f"❌ Silver cancellations: {df_silver_cancellations.count():,} satır")

print("\n📊 Purchase istatistikleri:")
df_silver_purchases.select("quantity", "unit_price", "total_price").describe().show()

print("\n🌍 Top 5 ülke (purchase):")
df_silver_purchases.groupBy("country").count().orderBy(col("count").desc()).show(5)

🛒 Silver purchases:    6,932 satır
❌ Silver cancellations: 170 satır

📊 Purchase istatistikleri:
+-------+------------------+-----------------+------------------+
|summary|          quantity|       unit_price|       total_price|
+-------+------------------+-----------------+------------------+
|  count|              6932|             6932|              6932|
|   mean|15.188545874206579|3.056449798038096|24.392345643392712|
| stddev|112.47392398579647|3.962623548706704| 86.38623210573566|
|    min|                 1|             0.06|              0.19|
|    max|              5184|            141.0|            1800.0|
+-------+------------------+-----------------+------------------+


🌍 Top 5 ülke (purchase):
+--------------+-----+
|       country|count|
+--------------+-----+
|United Kingdom| 6514|
|          EIRE|  119|
|        France|   80|
|      Portugal|   79|
|       Germany|   44|
+--------------+-----+
only showing top 5 rows



In [18]:
from delta import DeltaTable

dt = DeltaTable.forPath(spark, SILVER_PURCHASES_PATH)

print("📜 Tablo geçmişi (versiyonlar):")
dt.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

📜 Tablo geçmişi (versiyonlar):
+-------+-----------------------+---------+----------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                |
+-------+-----------------------+---------+----------------------------------------------------------------+
|2      |2026-05-10 14:26:36.128|WRITE    |{numFiles -> 1, numOutputRows -> 6932, numOutputBytes -> 241136}|
|1      |2026-05-10 14:05:55.981|WRITE    |{numFiles -> 1, numOutputRows -> 6932, numOutputBytes -> 241136}|
|0      |2026-05-10 13:59:57.401|WRITE    |{numFiles -> 1, numOutputRows -> 6932, numOutputBytes -> 241136}|
+-------+-----------------------+---------+----------------------------------------------------------------+



In [19]:
spark.stop()
print("✅ Spark kapatıldı")

✅ Spark kapatıldı
